# OpenDrumScribe Stage 2: Pure DSP Drum Transcriber 🎛️
抛弃不稳定的 AI 依赖，使用纯粹的数字信号处理（DSP）算法。通过频段隔离（低/中/高）与瞬态侦测，以极高的稳定性和速度直接生成标准 `.mid` 鼓谱。

**优势：** 安装极快，永不报错，直接避开 MIDI 128 键限制。

In [ ]:
# 1. Install Lightweight DSP Libraries (安装轻量级音频与 MIDI 库)
!pip install librosa scipy mido > /dev/null 2>&1
print("Libraries installed successfully! / 运行库安装完毕！")

In [ ]:
# 2. Upload Pure Drum Audio (上传你在 Stage 1 提取的纯净鼓声音频)
from google.colab import files
import os

if os.path.exists('drums_input.wav'):
    os.remove('drums_input.wav')

uploaded = files.upload()
original_file = next(iter(uploaded))
os.rename(original_file, 'drums_input.wav')
print("\nAudio ready for DSP analysis! / 音频已就绪！")

In [ ]:
# 3. Run DSP Algorithm & Generate MIDI (执行滤波、瞬态捕捉并输出完美 MIDI)
import librosa
import scipy.signal
from mido import Message, MidiFile, MidiTrack

print("Analyzing frequencies and transients...")
y, sr = librosa.load('drums_input.wav', sr=None, mono=True)

# 定义巴特沃斯滤波器
def apply_filter(data, cutoff, fs, btype, order=5):
    nyq = 0.5 * fs
    normal_cutoff = [c / nyq for c in cutoff] if isinstance(cutoff, list) else cutoff / nyq
    b, a = scipy.signal.butter(order, normal_cutoff, btype=btype, analog=False)
    return scipy.signal.filtfilt(b, a, data)

# 频段隔离：低频(底鼓)、中频(军鼓)、高频(踩镲)
kick_y = apply_filter(y, 150, sr, 'low')
snare_y = apply_filter(y, [150, 2000], sr, 'band')
hihat_y = apply_filter(y, 4000, sr, 'high')

# 瞬态侦测算法
kick_onsets = librosa.onset.onset_detect(y=kick_y, sr=sr, pre_max=10, post_max=10, pre_avg=20, post_avg=20, delta=0.1)
snare_onsets = librosa.onset.onset_detect(y=snare_y, sr=sr, pre_max=5, post_max=5, pre_avg=10, post_avg=10, delta=0.08)
hihat_onsets = librosa.onset.onset_detect(y=hihat_y, sr=sr, pre_max=3, post_max=3, pre_avg=5, post_avg=5, delta=0.05)

kick_times = librosa.frames_to_time(kick_onsets, sr=sr)
snare_times = librosa.frames_to_time(snare_onsets, sr=sr)
hihat_times = librosa.frames_to_time(hihat_onsets, sr=sr)

# 写入 MIDI 文件
mid = MidiFile()
track = MidiTrack()
mid.tracks.append(track)
ticks_per_sec = 960  # 设定标准 Tick 精度

events = []
for t in kick_times: events.append((t, 36))  # 36 = C2 (Kick)
for t in snare_times: events.append((t, 38)) # 38 = D2 (Snare)
for t in hihat_times: events.append((t, 42)) # 42 = F#2 (Hi-hat)
events.sort(key=lambda x: x[0])

prev_tick = 0
for time_sec, note in events:
    abs_tick = int(time_sec * ticks_per_sec)
    delta = max(0, abs_tick - prev_tick)
    track.append(Message('note_on', note=note, velocity=100, time=delta))
    track.append(Message('note_off', note=note, velocity=0, time=10))
    prev_tick = abs_tick + 10

mid.save('drums_output.mid')
print("MIDI generation complete! / 谱面生成完毕！")

In [ ]:
# 4. Download Standard MIDI (下载最终的鼓谱 MIDI)
if os.path.exists('drums_output.mid'):
    files.download('drums_output.mid')
else:
    print("Error generating MIDI.")